# Lesson 04 Lab — Closing the Loop: Train, Prune, Recover, and Re-evaluate

**Puzzle:** Why does a one-shot 70% mask often fail when the same target reached gradually can recover?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Pruning changes an optimization problem, not just a checkpoint file. The model must absorb a perturbation while the remaining weights adapt. A closed loop records the dense state, pruning event, recovery budget, best recovered metric, final mask, and rollback decision instead of reporting only the target sparsity.


## 0. Predict before running

1. Predict the immediate accuracy drop after one-shot 70% pruning.
2. Predict whether staged pruning will finish with a smaller or larger recovery gap.
3. Name the artifact needed to prove that zeros did not regrow.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A synthetic but separable classification dataset, a small MLP, one dense initialization, one-shot and staged pruning schedules, optimizer steps, masks reapplied after every update, and held-out accuracy form the experiment.

- Pruning is a state transition followed by constrained optimization.
- Masks must remain enforced during recovery.
- One-shot and gradual routes need equal initialization and declared budgets.


## 2. Derive the mechanism

Magnitude pruning projects weights onto a sparse support. An abrupt 70% projection can remove several co-adapted paths at once and move the loss far from the local basin. A staged schedule introduces smaller support changes followed by recovery. Because optimizers can regrow masked weights, the mask must be enforced after updates unless the parameterization guarantees zeros. Fairness requires both routes to begin from the identical dense checkpoint and consume a declared recovery budget.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 4
LESSON_TITLE = 'Closing the Loop: Train, Prune, Recover, and Re-evaluate'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260812
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | one-shot 70% magnitude pruning from the frozen dense checkpoint |
| Candidate | three-stage pruning to the same target with interleaved recovery |
| Held constant | initial checkpoint, dataset, split, optimizer family, total recovery steps, seed, and final sparsity |
| Measurements | dense accuracy, immediate drop, recovered accuracy, final sparsity, and best recovery step |
| Evidence | `pytorch-gpu` |

**Experiment:** Train one dense toy classifier, then compare one-shot and staged 70% pruning with equal recovery steps.


## 5. Read the experiment code

The lab trains one baseline and clones it before either pruning route. A mask helper chooses the global threshold and a recovery helper reapplies the mask after every optimizer step. Both routes consume the same total number of updates; the staged route only changes when support is removed. This makes schedule the independent variable.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
n, d, classes = 1536, 32, 4
x = torch.randn(n, d, device=DEVICE)
teacher = torch.randn(d, classes, device=DEVICE)
y = (x @ teacher + 0.2 * torch.randn(n, classes, device=DEVICE)).argmax(1)
train_x, val_x = x[:1200], x[1200:]
train_y, val_y = y[:1200], y[1200:]

def make_model():
    return nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, classes)).to(DEVICE)

def accuracy(model):
    model.eval()
    with torch.inference_mode():
        return float((model(val_x).argmax(1) == val_y).float().mean().item())

def train_steps(model, steps, masks=None, lr=0.03):
    model.train(); opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.8)
    for step in range(steps):
        idx = torch.arange(step * 96, step * 96 + 96, device=DEVICE) % train_x.shape[0]
        opt.zero_grad(); loss = F.cross_entropy(model(train_x[idx]), train_y[idx]); loss.backward(); opt.step()
        if masks:
            with torch.no_grad():
                for p, mask in masks.items(): p.mul_(mask)
    return float(loss.item())

def global_masks(model, amount):
    weights = [p for name, p in model.named_parameters() if "weight" in name]
    flat = torch.cat([p.detach().abs().flatten() for p in weights])
    k = int(flat.numel() * amount)
    threshold = torch.topk(flat, k, largest=False).values.max() if k else -1
    return {p: (p.detach().abs() > threshold).to(p.dtype) for p in weights}

def apply_masks(masks):
    with torch.no_grad():
        for p, mask in masks.items(): p.mul_(mask)

dense = make_model(); train_steps(dense, 140, lr=0.05)
dense_acc = accuracy(dense)
one = copy.deepcopy(dense); one_masks = global_masks(one, 0.70); apply_masks(one_masks)
one_immediate = accuracy(one); train_steps(one, 36, one_masks, lr=0.015); one_final = accuracy(one)
gradual = copy.deepcopy(dense); gradual_masks = None
trajectory = []
for stage, amount in enumerate((0.30, 0.50, 0.70), 1):
    gradual_masks = global_masks(gradual, amount); apply_masks(gradual_masks)
    immediate = accuracy(gradual)
    train_steps(gradual, 12, gradual_masks, lr=0.015)
    trajectory.append({"stage": stage, "target": amount, "immediate_accuracy": immediate, "recovered_accuracy": accuracy(gradual)})
gradual_final = accuracy(gradual)
final_sparsity = sum((p == 0).sum().item() for p in gradual_masks) / sum(p.numel() for p in gradual_masks)
metrics = {
    "dense_accuracy": dense_acc,
    "oneshot_immediate_accuracy": one_immediate,
    "oneshot_recovered_accuracy": one_final,
    "gradual_recovered_accuracy": gradual_final,
    "final_sparsity": final_sparsity,
    "recovery_steps_per_route": 36,
    "gradual_trajectory": trajectory,
}
analysis = (
    f"The dense toy classifier reached {dense_acc:.1%}. One-shot 70% pruning changed validation accuracy "
    f"immediately to {one_immediate:.1%} and recovered to {one_final:.1%} after 36 updates. The staged route "
    f"finished at {gradual_final:.1%} with {final_sparsity:.1%} zeros under the same update budget. "
    "This isolates the support trajectory on one synthetic task; it is not a universal schedule ranking."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Dense accuracy | 93.15% |
| One-shot immediate | 90.48% |
| One-shot recovered | 91.37% |
| Gradual recovered | 91.37% |
| Final sparsity | 69.97% |


## 7. Interpret rather than merely print

The dense toy classifier reached 93.2%. One-shot 70% pruning changed validation accuracy immediately to 90.5% and recovered to 91.4% after 36 updates. The staged route finished at 91.4% with 70.0% zeros under the same update budget. This isolates the support trajectory on one synthetic task; it is not a universal schedule ranking.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 4,
    "title": 'Closing the Loop: Train, Prune, Recover, and Re-evaluate',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'A pruning result is a trajectory with a support constraint and recovery budget, not a mask applied once.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 4,
  "title": "Closing the Loop: Train, Prune, Recover, and Re-evaluate",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260812
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "dense_accuracy": 0.9315476417541504,
    "oneshot_immediate_accuracy": 0.9047619104385376,
    "oneshot_recovered_accuracy": 0.9136905074119568,
    "gradual_recovered_accuracy": 0.9136905074119568,
    "final_sparsity": 0.6996527777777778,
    "recovery_steps_per_route": 36,
    "gradual_trajectory": [
      {
        "stage": 1,
        "target": 0.3,
        "immediate_accuracy": 0.9315476417541504,
        "recovered_accuracy": 0.9345238208770752
      },
      {
        "stage": 2,
        "target": 0.5,
        "immediate_accuracy": 0.9285714626312256,
        "recovered_accuracy": 0.9226190447807312
      },
      {
        "stage": 3,
        "targ

## 9. Make the bounded decision

> A pruning result is a trajectory with a support constraint and recovery budget, not a mask applied once.

**Acceptance/rollback:** Accept the pruned checkpoint only when held-out accuracy and sparsity both pass and the exact dense checkpoint remains available for rollback.

**Failure analysis:** A toy separable dataset can favor either schedule and does not predict ImageNet recovery. Comparing unequal training steps, learning rates, or data order also invalidates the causal claim. The lab establishes the control-loop mechanics, not a universal schedule ranking.


## 10. Extend the evidence

Repeat with several seeds and recovery budgets, plot accuracy immediately before and after each pruning event, and add a distillation term as a separately controlled intervention.

The full evidence boundary and references are in [`README.md`](README.md).
